# SentinelSleep — Pre-generate Therapeutic Audio Cache on Colab GPU

**Purpose:** Run `scripts/pregenerate_cache.py` on a Colab GPU (T4/L4/A100) to build
the `data/audio_cache/` directory.  Download the resulting zip and unpack it locally.

**Models used (ADR-014):**
- `facebook/musicgen-small` — ambient therapeutic music (300M params, ~1.2 GB)
- `facebook/audiogen-medium` — nature soundscapes (via Meta AudioCraft, ~1.5 GB)

**Before running:**
1. Set runtime to **GPU** → Runtime → Change runtime type → T4 GPU
2. Run all cells top-to-bottom without skipping

**Expected wall time:** ~10–15 min on T4 (3 music clips + 3 soundscapes + 10 mixes)

---

## Cell 1 — Clone the repo

In [ ]:
# If the repo is private, authenticate first:
#   !git config --global credential.helper store
#   !echo 'https://<YOUR_GH_PAT>:x-oauth-basic@github.com' > ~/.git-credentials

!git clone https://github.com/reddy-nithin/SentinalSleep.git
%cd SentinalSleep

# Pin to a specific commit so manifest.git_commit is meaningful.
# Change to a specific SHA if you want a reproducible build.
COMMIT = "main"
!git checkout {COMMIT}
!git log --oneline -3

## Cell 2 — Install dependencies

Colab ships with Python, pip, and a recent PyTorch — we install on top of that.
`audiocraft` is installed **after** torch so pip doesn't downgrade it.
The `sentinelsleep` package is made importable via `sys.path` (no build system needed).

In [ ]:
import subprocess, sys, os

# Core audio/ML deps
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.46.0",
    "diffusers>=0.30.0",
    "accelerate>=1.0.0",
    "librosa>=0.10.2",
    "scipy>=1.13.0",
    "soundfile>=0.12.1",
    "pydub>=0.25.1",
    "numpy>=1.26.0,<2.2.0",
], check=True)

# audiocraft must be installed AFTER torch to avoid torchaudio version conflicts.
# --no-deps prevents pip from downgrading torchaudio.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "audiocraft",
], check=True)

# Make sentinelsleep importable by adding src/ to the Python path.
# This avoids any build-system dependency (hatchling, pip editable install, etc.).
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Added {src_path} to sys.path")

# Verify the package is importable
import importlib
importlib.import_module("sentinelsleep.config")
print("\nAll dependencies installed and sentinelsleep importable.")

## Cell 3 — (Optional) Hugging Face authentication

Only needed if you hit rate-limit errors downloading models.  
Store your token in **Colab Secrets** (key icon in the left sidebar) as `HF_TOKEN`.  
Do NOT paste it directly in the notebook.

In [ ]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
        print("HF login successful")
    else:
        print("HF_TOKEN secret not set — proceeding without auth (may hit rate limits)")
except Exception as e:
    print(f"Could not load HF_TOKEN: {e} — proceeding without auth")

## Cell 4 — Run the cache builder

Runs three steps in sequence (memory-safe):
1. MusicGen-small → 3 × 60s ambient music clips
2. AudioGen-medium → 3 × 60s nature soundscapes  
3. Mixer → 5 mild + 5 severe intervention mixes

Each model is loaded, used, then unloaded before the next loads.  
Expected output: 16 WAV files + `manifest.json`.

> **Note:** `sys.path` from Cell 2 does not carry over to `!python` shell calls.
> We use `PYTHONPATH` env var so the script can import `sentinelsleep`.

In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = os.path.abspath("src")

result = subprocess.run(
    [sys.executable, "scripts/pregenerate_cache.py"],
    env=env,
    check=False,   # we print output regardless; check exit code manually below
)
if result.returncode != 0:
    raise RuntimeError(f"pregenerate_cache.py exited with code {result.returncode}")
print("\nCache build finished")

## Cell 5 — Verify the cache before downloading

In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = os.path.abspath("src")

subprocess.run(
    [sys.executable, "scripts/verify_cache.py", "--no-sha256"],
    env=env,
    check=True,
)
# --no-sha256 is fast enough for a quick sanity check here;
# the full SHA-256 check runs locally after download.

## Cell 6 — Zip and download

In [ ]:
import shutil, os

shutil.make_archive("audio_cache", "zip", "data", "audio_cache")
size_mb = os.path.getsize("audio_cache.zip") / 1_048_576
print(f"Zipped → audio_cache.zip  ({size_mb:.1f} MB)")

In [ ]:
from google.colab import files
files.download("audio_cache.zip")

---
## After download — run these commands locally

```bash
# 1. Unpack into your local repo
cd /path/to/SentinalSleep
unzip -o ~/Downloads/audio_cache.zip -d data/

# 2. Full integrity check (includes SHA-256)
uv run python scripts/verify_cache.py

# 3. Confirm all tests still pass
uv run pytest tests/ -q
```

If `verify_cache.py` exits 0, Phase 3 is complete and you can start Phase 4 (Orchestration).

### Troubleshooting

| Issue | Fix |
|-------|-----|
| `audiocraft` import error | Cell 2 install failed — re-run Cell 2 |
| `sentinelsleep` not found in Cell 4/5 | Re-run Cell 2 (sets PYTHONPATH) |
| HF rate limit / 401 | Set `HF_TOKEN` in Colab Secrets, re-run Cell 3 |
| AudioGen OOM on T4 | Unlikely (only 1.5 GB); if it happens, switch to L4/A100 |
| Missing WAVs after verify | Re-run Cell 4 with `--skip-music` or `--skip-soundscapes` flags |
| `audio_cache.zip` download fails | Use Files panel (folder icon) to download manually |
